# ToDo:
- Interpolation feature

# Directivity DataFrame to MISUKA Binary

Converts a directivity-measurement DataFrame into the binary file read by
the MISUKA `speaker` plugin.

## Input format (what future users should bring)

A pandas DataFrame with:
  - **Index:** MultiIndex `(phi_deg, theta_deg)`, integer values.
    `phi_deg` in `[0, 360)`, `theta_deg` in `[0, 180]`.
  - **Columns:** float frequencies in Hz (e.g. `1000.0`).
  - **Cells:** complex *or* real magnitudes. `np.abs()` is applied unconditionally,
    so both work.

Step 1 of this notebook is one example pipeline (Genelec CSV → DataFrame).
Future users replace step 1 with their own loader and run steps 2–7 unchanged.

## Output format

Little-endian binary:
```
Offset  Bytes        Type         Meaning
------  -----------  -----------  ---------------------------
   0    2            uint16       n_freq
   2    2            uint16       H (theta resolution)
   4    2            uint16       W (phi resolution)
   6    4*n_freq     float32[]    frequency list (Hz)
 ...    4*H*W*n      float32[]    pixel data (H, W, n_freq), row-major
```

directivity values layout: `data[theta, phi, freq]` (theta is rows, phi is columns).

In [1]:
import struct
from pathlib import Path

import numpy as np
import pandas as pd

## Parameters

In [2]:
OUTPUT_PATH = "../directivity_textures/genelec8020_directivity.bin"

# Frequencies (Hz) to include as channels in the output file.
# Each must exist as a column in the input DataFrame.
FREQUENCIES = [250, 500, 1000, 2000, 4000]

# Texture resolution. Theta = polar angle [0, 180] (rows), Phi = azimuth [0, 360) (cols).
H = 181  # theta
W = 360  # phi

# Genelec-CSV-specific (only used in step 1)
CSV_PATH   = "Genelec8020_1x1_64442_MPS_front_pole.csv"
CSV_N_ROWS = 64442

## 1. Load measurements into the standard DataFrame

**This step is source-specific. Users with a different dataset have to insert a cell for their own dataset and skip this cell.The outcome has to be a DataFrame
matching the schema described at the top of this notebook** 

(MultiIndex `(phi_deg,theta_deg)`, float frequency columns, complex magnitudes or real magnitudes).

The Genelec CSV has positions encoded as `'PXXXTYYY'` strings and values as complex strings (`'4.4 + 16.0i'`). We parse both into native pandas
types, then convert the index to a MultiIndex.

In [3]:
def to_complex_safe(val):
    """Convert a single CSV cell to complex number."""
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        val = val.strip()
        if val == "NaN - NaNi":
            return np.nan
        val = val.replace("i", "j").replace(" ", "")
        try:
            return complex(val)
        except ValueError:
            return np.nan
    return val

df = pd.read_csv(
    CSV_PATH,
    delimiter=",",
    decimal=".",
    header=0,
    index_col=0,
    nrows=CSV_N_ROWS,
    skipinitialspace=True,
)



df = df.map(to_complex_safe)
df = df.dropna(axis=1, how="all")

df.columns = pd.to_numeric(df.columns, errors="raise")



# Convert 'PXXXTYYY' index to MultiIndex (phi_deg, theta_deg) with integer values.
phi_deg   = df.index.str.slice(1, 4).astype(int)
theta_deg = df.index.str.slice(5, 8).astype(int)
df.index  = pd.MultiIndex.from_arrays([phi_deg, theta_deg], names=["phi_deg", "theta_deg"])

print(f"Loaded DataFrame: {len(df)} positions x {len(df.columns)} frequencies")
print(f"phi_deg range:   [{df.index.get_level_values('phi_deg').min()}, "
      f"{df.index.get_level_values('phi_deg').max()}]")
print(f"theta_deg range: [{df.index.get_level_values('theta_deg').min()}, "
      f"{df.index.get_level_values('theta_deg').max()}]")
print(f"freq range:      [{df.columns.min()}, {df.columns.max()}] Hz")
df.head()

Loaded DataFrame: 64442 positions x 30 frequencies
phi_deg range:   [0, 359]
theta_deg range: [0, 180]
freq range:      [20.0, 20000.0] Hz


20.0                  31.5     \
phi_deg theta_deg                                              
0       0          4.407277+16.052855j -81.871808-72.987188j   
        1          4.430241+16.059229j -81.943978-72.831609j   
1       1          4.430526+16.059250j -81.945347-72.829561j   
2       1          4.430806+16.059269j -81.946702-72.827549j   
3       1          4.431080+16.059287j -81.948040-72.825575j   

                                  40.0                     50.0     \
phi_deg theta_deg                                                    
0       0         -159.751517+233.789089j  590.886252+1505.674253j   
        1         -159.332343+234.134272j  594.626282+1504.190695j   
1       1         -159.328450+234.137099j  594.646291+1504.182896j   
2       1         -159.324663+234.139836j  594.665301+1504.175496j   
3       1         -159.320984+234.142482j  594.683304+1504.168498j   

                                    63.0                      80.0     \
phi_deg theta_deg                                                       
0       0          2774.259344+1140.102858j  2731.612138-1772.992986j   
        1          2777.432212+1132.295718j  2725.521464-1782.337673j   
1       1          2777.441392+1132.272542j  2725.515373-1782.347027j   
2       1          2777.449705+1132.251505j  2725.510989-1782.353772j   
3       1          2777.457149+1132.232615j  2725.508314-1782.357905j   

                                   100.0                     125.0    \
phi_deg theta_deg                                                      
0       0         -267.186504-3226.556713j -3448.125272-1234.293769j   
        1         -281.025450-3225.385274j -3454.867655-1215.300881j   
1       1         -281.024015-3225.385442j -3454.864321-1215.310303j   
2       1         -281.018658-3225.385950j -3454.859074-1215.325163j   
3       1         -281.009382-3225.386799j -3454.851914-1215.345456j   

                                    160.0                     200.0    ...  \
phi_deg theta_deg                                                      ...   
0       0         -2319.411850+2747.499086j  1963.604172+3177.852308j  ...   
        1         -2299.619988+2764.092762j  1993.854399+3158.803643j  ...   
1       1         -2299.631339+2764.083371j  1993.837864+3158.814093j  ...   
2       1         -2299.648410+2764.069218j  1993.812577+3158.830112j  ...   
3       1         -2299.671196+2764.050310j  1993.778547+3158.851694j  ...   

                                    2500.0                    3150.0   \
phi_deg theta_deg                                                       
0       0          -673.856499-4188.676059j -2988.066596-3266.020280j   
        1         -1253.037596-4066.882521j -3392.252023-2926.437854j   
1       1         -1252.805974-4066.979178j -3392.345103-2926.408073j   
2       1         -1252.403638-4067.124449j -3392.330046-2926.480683j   
3       1         -1251.830704-4067.318228j -3392.206915-2926.655566j   

                                    4000.0                    5000.0   \
phi_deg theta_deg                                                       
0       0          3548.615973-1168.721676j  1043.457641-3343.308653j   
        1          3355.834278-1693.913422j   347.724024-3553.664219j   
1       1          3356.017970-1693.832122j   349.170782-3553.599095j   
2       1          3356.262406-1693.616000j   350.807155-3553.495033j   
3       1          3356.567499-1693.265056j   352.632485-3553.351813j   

                                    6300.0                    8000.0   \
phi_deg theta_deg                                                       
0       0          3605.752800- 501.319863j  4059.742121- 192.206141j   
        1          3379.041027-1389.218703j  3861.971367-1366.970620j   
1       1          3380.079161-1387.034611j  3862.903704-1364.894851j   
2       1          3381.205333-1384.623103j  3863.934885-1362.506778j   
3       1          3382.418626-1381.984900j  3865.063875-1359.807

## 2. Validate that all requested frequencies are present

In [4]:
available_freqs = df.columns.to_numpy(dtype=float)

missing = [f for f in FREQUENCIES if f not in available_freqs]
if missing:
    raise ValueError(
        f"Requested frequencies not found in DataFrame: {missing}. "
        f"Available: {available_freqs.tolist()}"
    )

print(f"All {len(FREQUENCIES)} requested frequencies found.")

All 5 requested frequencies found.


## 3. Magnitude, normalization, scatter into (H, W, n_freq) array

For each requested frequency:
  - Apply `np.abs()` — handles complex inputs (returns `|z|`) and real inputs
    (returns `|x|`, no-op for non-negative magnitudes).
  - Normalize by the per-frequency peak so each band lies in `[0, 1]`.
  - Put each `(phi, theta)` sample into the output array.

In [5]:
NORMALIZATION = 'on_axis_per_frequency'
# NORMALIZATION = 'per_frequency_peak'
# NORMALIZATION = 'already_normalized'


n_freq = len(FREQUENCIES)
data   = np.zeros((H, W, n_freq), dtype=np.float32)

phi_idx   = df.index.get_level_values("phi_deg").to_numpy()
theta_idx = df.index.get_level_values("theta_deg").to_numpy()

for out_idx, freq in enumerate(FREQUENCIES):
    magnitudes = np.abs(df[freq].to_numpy())

    if NORMALIZATION == 'on_axis_per_frequency':
        denominator = np.abs(df.loc[(0, 0), freq])
    elif NORMALIZATION == 'per_frequency_peak':
        denominator = np.max(magnitudes)
    elif NORMALIZATION == 'already_normalized':
        denominator = 1.0
    
    if denominator <= 0:
        denominator += np.finfo('float64').eps
    
    normalized_magnitudes = magnitudes / denominator 
    data[theta_idx, phi_idx, out_idx] = normalized_magnitudes

print(f"Built texture array: shape={data.shape} (H=theta, W=phi, n_freq)")

Built texture array: shape=(181, 360, 5) (H=theta, W=phi, n_freq)


## 4. Pole handling

At `theta=0` and `theta=180`, all phi values represent the same physical point.
The DataFrame typically has one measurement per pole; replicate it across all
azimuths. If a pole has no nonzero measurement (incomplete data), this raises
`IndexError` — fail loudly rather than silently use zero.

In [6]:
for freq_idx in range(n_freq):
    front_nonzero = data[0, :, freq_idx][data[0, :, freq_idx] != 0]
    data[0, :, freq_idx] = front_nonzero[0]

    back_nonzero = data[H - 1, :, freq_idx][data[H - 1, :, freq_idx] != 0]
    data[H - 1, :, freq_idx] = back_nonzero[0]

print("Pole handling complete (front and back poles filled across all azimuths).")

Pole handling complete (front and back poles filled across all azimuths).


## 5. Write the binary file

Layout: 6-byte header (`uint16` × 3) + frequency list (`float32` × n_freq) +
pixel data (`float32` × H × W × n_freq), little-endian, row-major.

In [7]:
freqs_arr = np.asarray(FREQUENCIES, dtype=np.float32)

with open(OUTPUT_PATH, "wb") as f:
    f.write(struct.pack("<HHH", n_freq, H, W)) #writes the header, which specifies the data order
    f.write(freqs_arr.tobytes()) #writes the frequency list
    f.write(data.tobytes()) # writes the actual texture data

print(f"Wrote {OUTPUT_PATH}")

Wrote ../directivity_textures/genelec8020_directivity.bin


## 6. Verify I - Using Python

Re-read the file, check size matches the formula, and confirm the data round-trips bit-exactly.

In [8]:
expected_size = 6 + 4 * n_freq + 4 * H * W * n_freq
actual_size   = Path(OUTPUT_PATH).stat().st_size
assert actual_size == expected_size, f"size mismatch: {actual_size} != {expected_size}"

with open(OUTPUT_PATH, "rb") as f:
    n_freq_r, H_r, W_r = struct.unpack("<HHH", f.read(6))
    freqs_r  = np.frombuffer(f.read(4 * n_freq_r), dtype=np.float32)
    pixels_r = np.frombuffer(f.read(4 * H_r * W_r * n_freq_r), dtype=np.float32)
    data_r   = pixels_r.reshape(H_r, W_r, n_freq_r)

assert (n_freq_r, H_r, W_r) == (n_freq, H, W), "header mismatch"
assert np.array_equal(freqs_r, freqs_arr),     "frequency list mismatch"
assert np.array_equal(data_r, data),           "pixel data mismatch"

print(f"OK: {actual_size} bytes")
print(f"  header:      n_freq={n_freq_r}, H={H_r}, W={W_r}")
print(f"  frequencies: {freqs_r.tolist()}")
print(f"  pixel range: [{data_r.min():.4f}, {data_r.max():.4f}]")

OK: 1303226 bytes
  header:      n_freq=5, H=181, W=360
  frequencies: [250.0, 500.0, 1000.0, 2000.0, 4000.0]
  pixel range: [0.0464, 1.0544]


## 7. Verify I - Using Misuka

Create a speaker inside a scene,pass the directivity tensor and compare it to the original

In [9]:
import mitsuba as mi
mi.set_variant("scalar_acoustic") 


scene_dict = {
    'type': 'scene',
    'speaker': {
        'type': 'sphere',
        'emitter': {
            'type': 'speaker',
            'radiance': 1.0,
            'directivity_file': OUTPUT_PATH,  
        },
    },
}
scene = mi.load_dict(scene_dict)
params = mi.traverse(scene)
tensor = params['speaker.emitter.directivity_tensor']
tensor_data = np.array(tensor)
if np.array_equal(data,tensor_data):
    print("The texture array was perfectly reconstructed in the speaker.cpp")

The texture array was perfectly reconstructed in the speaker.cpp
